No WS4 command here uses shell magic: every one goes through this helper, which is also
how you would drive the command line from any script.

This one is a **command-line** session.


In [1]:
import subprocess


def run(command: str) -> None:
    """Run a WS4 command, printing what it printed."""
    result = subprocess.run(command, shell=True, text=True, capture_output=True)
    print((result.stdout or result.stderr).strip())
    result.check_returncode()

# Route products to mills

The sample estate ships a road network with two mills at opposite ends, so the least-haul-cost
destination differs across the estate. This notebook routes a schedule and reads the accounts
the routing produces, then constrains the haul cost in a search.

## The network

One access vertex per grid row, a road along them, a spur to `destination.MILL1` at the west end
and one to `destination.MILL2` at the east end. `examples/estate/route.yaml` names the four
tables and defines the route accounts: haul and volume per mill, and a total haul.

In [2]:
from pathlib import Path

import pandas as pd

work = Path("tmp/examples/routes")
work.mkdir(parents=True, exist_ok=True)
schedule = pd.DataFrame({"block_id": [1, 2, 9, 10, 57, 58, 63, 64], "treatment": ["CC"] * 8, "period": [0] * 8})
(work / "schedule.csv").write_text(schedule.to_csv(index=False))

87

In [3]:
run("ws4 simulate --dataset tmp/examples/estate/compiled --schedule tmp/examples/routes/schedule.csv"
    " --years 60 --topology examples/estate/topology.csv"
    " --route-config examples/estate/route.yaml --out tmp/examples/routes/run")

Simulation complete → tmp/examples/routes/run


In [4]:
routed = pd.read_csv(work / "run" / "route_values.csv")
print(routed.groupby("destination")[["volume", "length_m", "haul_cost"]].sum().to_string())
print()
print(routed[["block_id", "destination", "volume", "length_m", "haul_cost"]].to_string(index=False))

                        volume  length_m  haul_cost
destination                                        
destination.MILL1  3423.514286    8000.0        0.4
destination.MILL2  5370.300000   12000.0        0.6

 block_id       destination      volume  length_m  haul_cost
        1 destination.MILL1 1918.714286    4000.0       0.20
        2 destination.MILL1 1504.800000    4000.0       0.20
        9 destination.MILL2 1272.000000    4000.0       0.20
       10 destination.MILL2 1932.000000    4000.0       0.20
       57 destination.MILL2  243.000000    1000.0       0.05
       58 destination.MILL2  294.300000    1000.0       0.05
       63 destination.MILL2  684.000000    1000.0       0.05
       64 destination.MILL2  945.000000    1000.0       0.05


Every load went to the cheaper mill for its block: the choice is the sum of the segment haul
rates along the route, so a block near the middle can be closer to one mill in distance and still
cheaper to send to the other.

In [5]:
values = pd.read_csv(work / "run" / "account_values.csv")
print(values[(values.year == 0) & values.account.str.startswith(("route.", "product.Mill"))].to_string(index=False))

 year             account       value
    0 product.Mill1.Yield 3423.514286
    0 product.Mill2.Yield 5370.300000
    0    route.Mill1.haul  684.702857
    0    route.Mill2.haul  749.115000
    0    route.Total.haul 1433.817857


## Constrain the haul cost in a search

In [6]:
(work / "targets.yaml").write_text(
    "targets:\n"
    "- account: product.Harvested.CC\n"
    "  year: 0\n"
    "  minimum: 20000\n"
    "  period_width: 10\n"
    "- account: route.Total.haul\n"
    "  year: 0\n"
    "  maximum: 3000\n"
    "  period_width: 10\n"
)

run("ws4 optimize --dataset tmp/examples/estate/compiled --targets tmp/examples/routes/targets.yaml"
    " --years 100 --iterations 20000 --route-config examples/estate/route.yaml --threads 1"
    " --out tmp/examples/routes/search")

                              Optimization Summary                              
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ key                 ┃ value                                                  ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ incumbent_objective │ 0.0                                                    │
│ iterations          │ 20000                                                  │
│ seed                │ 42                                                     │
│ metaheuristic       │ sa                                                     │
│ starts              │ 1                                                      │
│ blocks_scheduled    │ 36                                                     │
│ operators           │ ['toggle', 'shift', 'swap', 'lns', 'cluster', 'merge', │
│                     │ 'split']                                               │
│ status              │ ok  

In [7]:
print(pd.read_csv(work / "search" / "report_targets.csv").to_string(index=False))
print()
print(pd.read_csv(work / "search" / "route_cost_report.csv").head(3).to_string(index=False))

             account  year        value  minimum  maximum  penalty status
product.Harvested.CC     0 20302.671429  20000.0      NaN      0.0    met
    route.Total.haul     0  2775.829286      NaN   3000.0      0.0    met

 PERIOD  YEAR  PERIODWIDTH  LENGTH  BUILTLENGTH  BUILD_EXPENSE  HAUL_EXPENSE  MAINTAIN_EXPENSE         TOTAL       VOLUME  AVERAGEHAULDIST  AVERAGEBUILDCOST  AVERAGEHAULCOST  AVERAGEMAINTCOST
      0     0            1  8000.0       8000.0       240000.0   2775.829286           24000.0 266775.829286 20302.671429      2734.447332         11.821104         0.136722          1.182110
      4     4            1  1000.0          0.0            0.0     66.957143            3000.0   3066.957143  1339.142857      1000.000000          0.000000         0.050000          2.240239
      6     6            1  4000.0          0.0            0.0    403.200000           12000.0  12403.200000  2016.000000      4000.000000          0.000000         0.200000          5.952381


## Where to look next

* `docs/howto/route-products.rst` walks through the tables and the route configuration.
* `docs/reference/roads.rst` specifies the network, the routing and the route outputs.